## Data Cleaning Automation Assignment

### Question 10 (prior questions 1-9 were conceptual and only in MyEducator)

**Goal**: Create a function named wrangle_basic(df) that cleans categorical text fields to eliminate data quality issues caused by inconsistent data entry.

Your function requirements:

* Identify which categorical columns have data quality issues
* Create cleaned versions of problematic columns (append "_clean" to column names; do not overwrite originals)
* Ensure that semantically identical values are represented consistently
* Preserve the original number of rows
* Use logic that would generalize to new data with similar patterns

**Hint**: Start by exploring the unique values in categorical columns to identify patterns of inconsistency.

Check question: After running your function, how many rows are labeled as "failed" in delivery_status_clean?

In [ ]:
# Question 10

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load the data
df = pd.read_csv('/Users/waylansmac/Desktop/455/last_mile_delivery_stops_1000.csv')

def wrangle_basic(df):
    """
    Clean categorical text fields to eliminate data quality issues.
    Creates cleaned versions with '_clean' suffix.
    """
    df = df.copy()
    
    # Identify categorical columns (object dtype)
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    
    # Columns to clean (those with text inconsistencies)
    cols_to_clean = ['hub', 'delivery_zone', 'delivery_note', 'customer_type', 
                     'priority_level', 'weather', 'delivery_status', 'failure_reason']
    
    for col in cols_to_clean:
        if col in df.columns:
            # Create cleaned column
            cleaned_col = col + '_clean'
            
            # Strip whitespace, convert to lowercase, standardize
            df[cleaned_col] = (df[col]
                              .astype(str)
                              .str.strip()
                              .str.lower()
                              .str.replace(r'\s+', ' ', regex=True)  # normalize whitespace
                              .replace('nan', np.nan))  # convert 'nan' strings back to NaN
    
    return df

# Apply the function
df = wrangle_basic(df)

# Check question: How many rows are labeled as "failed" in delivery_status_clean?
failed_count = (df['delivery_status_clean'] == 'failed').sum()
print(f"Question 10 Check: Number of 'failed' deliveries: {failed_count}")
print(f"Unique values in delivery_status_clean: {df['delivery_status_clean'].unique()}")


### Question 11

**Goal**: Create a function named add_datetime_features(df) that converts messy datetime strings into usable datetime objects and creates time-based analytical features.

Your function requirements:

* Parse datetime information from text columns that contain date/time data
  * Handle inconsistent formatting (different date orders, 12/24-hour time, timezone tokens)
  * Successfully parse at least these columns: stop_datetime_raw, scheduled_window_start_raw
  * Create parsed versions with appropriate naming to distinguish from originals

* Engineer time-based features for analysis:
  * Day of week indicator (numeric, starting with Monday)
  * Weekend indicator (binary: weekend vs weekday)
  * Delivery lateness metric that compares actual arrival time to the scheduled window

**Hint**: The actual_arrival_min column contains minutes from midnight. Your lateness metric should be comparable to this scale and never negative.

**Check question**: After running your function (on the output of Question 10), what is the mean of your lateness metric, rounded to 2 decimals?

In [ ]:
# Question 11

def add_datetime_features(df):
    """
    Parse messy datetime strings and create time-based analytical features.
    """
    df = df.copy()
    
    # Parse datetime columns that have inconsistent formatting
    datetime_cols = ['stop_datetime_raw', 'scheduled_window_start_raw']
    
    for col in datetime_cols:
        if col in df.columns:
            # Use pandas to_datetime with infer_datetime_format for flexibility
            parsed_col = col.replace('_raw', '_parsed')
            df[parsed_col] = pd.to_datetime(df[col], errors='coerce', infer_datetime_format=True)
    
    # Use the parsed scheduled_window_start for feature engineering
    if 'scheduled_window_start_parsed' in df.columns:
        # Day of week (Monday=0, Sunday=6)
        df['day_of_week'] = df['scheduled_window_start_parsed'].dt.dayofweek
        
        # Weekend indicator (Saturday=5, Sunday=6)
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        
        # Extract scheduled window in minutes from midnight
        df['scheduled_min'] = (df['scheduled_window_start_parsed'].dt.hour * 60 + 
                               df['scheduled_window_start_parsed'].dt.minute)
        
        # Delivery lateness metric: difference between actual arrival and scheduled time
        # Using scheduled_window_min column (which appears to be planned arrival) 
        # compared to actual_arrival_min
        if 'actual_arrival_min' in df.columns and 'scheduled_window_min' in df.columns:
            # Lateness = actual - scheduled (never negative, using max with 0)
            df['lateness_min'] = np.maximum(0, df['actual_arrival_min'] - df['scheduled_window_min'])
    
    return df

# Apply the function
df = add_datetime_features(df)

# Check question: Mean of lateness metric
lateness_mean = df['lateness_min'].mean()
print(f"Question 11 Check: Mean lateness = {lateness_mean:.2f} minutes")
print(f"Sample datetime parsing results:")
print(df[['stop_datetime_raw', 'stop_datetime_parsed', 'day_of_week', 'is_weekend']].head())


### Question 12

**Goal**: Create a function named bin_rare_categories(df, cols=None, min_prop=0.05, suffix='_binned') that consolidates infrequent categories to reduce cardinality.

Your function requirements:

* Accept flexible column specification:
  * If `cols=None` (default), process all categorical columns in the DataFrame
  * If `cols` is a string, process that single column
  * If `cols` is a list, process those specific columns
  
* Reduce category cardinality by grouping rare values:
  * Calculate the frequency of each category as a proportion of total rows
  * Categories that appear too infrequently should be consolidated into a single group
  * The threshold for "infrequent" is controlled by the `min_prop` parameter (default: 0.05)
  
* Create new binned columns:
  * Add a suffix to distinguish binned columns from originals (controlled by `suffix` parameter, default: '_binned')
  * Preserve all rows (no filtering)

**Hint**: Common practice is to label consolidated rare categories as "Other"

**Check question**: After running your function on delivery_zone_clean (from Question 11 output) with default parameters, how many unique categories exist in the resulting binned column?

In [ ]:
# Question 12

def bin_rare_categories(df, cols=None, min_prop=0.05, suffix='_binned'):
    """
    Consolidate infrequent categories to reduce cardinality.
    Categories below min_prop threshold are grouped as 'Other'.
    """
    df = df.copy()
    
    # If cols is None, process all categorical columns
    if cols is None:
        cols = df.select_dtypes(include=['object']).columns.tolist()
    elif isinstance(cols, str):
        cols = [cols]
    
    for col in cols:
        if col not in df.columns:
            continue
            
        # Calculate frequency proportions
        value_counts = df[col].value_counts(dropna=False)
        total_rows = len(df)
        proportions = value_counts / total_rows
        
        # Identify rare categories (below threshold)
        rare_categories = proportions[proportions < min_prop].index.tolist()
        
        # Create binned column
        binned_col = col + suffix
        df[binned_col] = df[col].copy()
        
        # Replace rare categories with 'Other'
        df.loc[df[col].isin(rare_categories), binned_col] = 'Other'
    
    return df

# Apply the function to delivery_zone_clean
df = bin_rare_categories(df, cols='delivery_zone_clean')

# Check question: How many unique categories in delivery_zone_clean_binned?
unique_count = df['delivery_zone_clean_binned'].nunique()
print(f"Question 12 Check: Unique categories in delivery_zone_clean_binned: {unique_count}")
print(f"Value counts:")
print(df['delivery_zone_clean_binned'].value_counts())


### Question 13

**Goal**: Create a function named transform_skew(df, features=None, suffix='_skewfix') that reduces skew in numeric columns by automatically selecting the best transformation.

Your function requirements:

* Accept flexible feature specification:
  * If `features=None` (default), process all numeric non-boolean columns
  * If `features` is a string, process that single column
  * If `features` is a list, process those specific columns

* Handle data with varied characteristics:
  * Some columns may contain negative values, zeros, or missing values
  * Your transformations must handle these gracefully without errors
  * Consider using a shift strategy for transforms that require non-negative inputs

* Evaluate multiple transformation strategies:
  * Test several monotonic transformations including no transformation as a baseline
  * Include both traditional power transformations AND advanced statistical transformations
  * Research the Yeo-Johnson transformation - a powerful method that handles negative values natively
  * Select the transformation that minimizes the absolute value of skewness
  * Implement a consistent tie-breaking strategy if multiple transformations produce similar results

* Create new columns with appropriate naming:
  * Use the `suffix` parameter to distinguish transformed columns from originals (default: '_skewfix')
  * Preserve all rows and original columns

**Hints**: 
- Skewness can be calculated with `.skew(skipna=True)`
- For transforms requiring non-negative inputs: consider shifting data so minimum becomes zero
- The Yeo-Johnson transformation is available in scipy.stats and works with negative values
- Common power transformations: logarithmic (log1p), square root, cube root
- When comparing transformations, focus on which produces skewness closest to zero

**Check question**: After running your function on distance_from_prev_mi (from Question 12 output) with default parameters, what is the skewness of the transformed column, rounded to 3 decimals?

In [ ]:
# Question 13

from scipy.stats import yeojohnson

def transform_skew(df, features=None, suffix='_skewfix'):
    """
    Reduce skew in numeric columns by automatically selecting the best transformation.
    Tests multiple transformations and selects the one that minimizes absolute skewness.
    """
    df = df.copy()
    
    # If features is None, process all numeric non-boolean columns
    if features is None:
        # Get numeric columns, excluding boolean columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        # Exclude boolean-like columns (0/1 binary)
        features = [col for col in numeric_cols 
                   if df[col].dropna().nunique() > 2]
    elif isinstance(features, str):
        features = [features]
    
    for col in features:
        if col not in df.columns:
            continue
        
        # Get non-null values for transformation testing
        data = df[col].dropna()
        
        if len(data) == 0:
            continue
        
        # Dictionary to store transformations and their resulting skewness
        transformations = {}
        
        # 1. No transformation (baseline)
        transformations['none'] = (data, abs(data.skew()))
        
        # 2. Log transformation (for positive values, shift if needed)
        if data.min() >= 0:
            transformations['log1p'] = (np.log1p(data), abs(np.log1p(data).skew()))
        else:
            # Shift to make all values non-negative
            shifted = data - data.min()
            transformations['log1p'] = (np.log1p(shifted), abs(np.log1p(shifted).skew()))
        
        # 3. Square root (for non-negative values, shift if needed)
        if data.min() >= 0:
            transformations['sqrt'] = (np.sqrt(data), abs(np.sqrt(data).skew()))
        else:
            shifted = data - data.min()
            transformations['sqrt'] = (np.sqrt(shifted), abs(np.sqrt(shifted).skew()))
        
        # 4. Cube root (works with negative values)
        transformations['cbrt'] = (np.cbrt(data), abs(np.cbrt(data).skew()))
        
        # 5. Square (inverse transformation)
        transformations['square'] = (data ** 2, abs((data ** 2).skew()))
        
        # 6. Yeo-Johnson transformation (handles negative values natively)
        try:
            transformed_yj, _ = yeojohnson(data)
            transformations['yeojohnson'] = (pd.Series(transformed_yj, index=data.index), 
                                            abs(pd.Series(transformed_yj).skew()))
        except:
            pass
        
        # Select transformation with minimum absolute skewness
        # Use consistent tie-breaking: alphabetical order of transformation name
        best_transform = min(transformations.items(), 
                           key=lambda x: (x[1][1], x[0]))
        
        # Apply the best transformation to the full column
        transformed_col = col + suffix
        
        if best_transform[0] == 'none':
            df[transformed_col] = df[col]
        elif best_transform[0] == 'log1p':
            if df[col].min() >= 0:
                df[transformed_col] = np.log1p(df[col])
            else:
                df[transformed_col] = np.log1p(df[col] - df[col].min())
        elif best_transform[0] == 'sqrt':
            if df[col].min() >= 0:
                df[transformed_col] = np.sqrt(df[col])
            else:
                df[transformed_col] = np.sqrt(df[col] - df[col].min())
        elif best_transform[0] == 'cbrt':
            df[transformed_col] = np.cbrt(df[col])
        elif best_transform[0] == 'square':
            df[transformed_col] = df[col] ** 2
        elif best_transform[0] == 'yeojohnson':
            df[transformed_col], _ = yeojohnson(df[col].fillna(df[col].median()))
    
    return df

# Apply the function to distance_from_prev_mi
df = transform_skew(df, features='distance_from_prev_mi')

# Check question: Skewness of the transformed column
transformed_skew = df['distance_from_prev_mi_skewfix'].skew()
print(f"Question 13 Check: Skewness of distance_from_prev_mi_skewfix: {transformed_skew:.3f}")
print(f"Original skewness: {df['distance_from_prev_mi'].skew():.3f}")


### Question 14

**Goal**: Create a function named impute_missing(df, features=None, group_cols=None) that fills in missing values without losing any data rows.

Your function requirements:

* Accept flexible feature specification:
  * If `features=None` (default), impute all columns that contain missing values
  * If `features` is a string, impute that single column
  * If `features` is a list, impute only those specific columns

* Accept flexible grouping specification:
  * If `group_cols=None` (default), automatically select logical grouping columns from the data
  * Look for cleaned categorical columns that represent natural groups (e.g., location, category indicators)
  * If `group_cols` is a list, use those specific columns for grouping

* Implement intelligent imputation strategies:
  * Use different strategies for numeric vs categorical data
  * Apply group-based imputation first (calculate statistics within groups)
  * Fall back to global imputation when group-based values are unavailable
  * Ensure deterministic results (same input always produces same output)

* Preserve all rows:
  * Never drop rows or columns
  * All missing values must be filled

**Hints**:
- For numeric data: central tendency measures like median work well
- For categorical data: most frequent values (mode) are appropriate
- Group-based imputation: calculate statistics separately for each group, then use those to fill within-group missing values
- Fallback strategy: if a group has all missing values, use the overall (global) statistic

**Check question**: After running your function on the output of Question 13, what is the mean of cargo_temp_f for records where hub_clean == "denver-east", rounded to 2 decimal places?

In [ ]:
# Question 14

def impute_missing(df, features=None, group_cols=None):
    """
    Fill missing values using group-based imputation with global fallback.
    Uses median for numeric, mode for categorical.
    """
    df = df.copy()
    
    # If features is None, impute all columns with missing values
    if features is None:
        features = df.columns[df.isnull().any()].tolist()
    elif isinstance(features, str):
        features = [features]
    
    # If group_cols is None, automatically select cleaned categorical columns
    if group_cols is None:
        # Look for cleaned categorical columns (ending with '_clean')
        clean_cols = [col for col in df.columns if col.endswith('_clean')]
        # Select a few key grouping columns
        group_cols = [col for col in ['hub_clean', 'delivery_zone_clean', 
                                       'customer_type_clean', 'weather_clean'] 
                     if col in clean_cols]
    
    for col in features:
        if col not in df.columns or df[col].isnull().sum() == 0:
            continue
        
        # Determine if numeric or categorical
        is_numeric = pd.api.types.is_numeric_dtype(df[col])
        
        if is_numeric:
            # Use median for numeric data
            if group_cols and len(group_cols) > 0:
                # Group-based imputation
                for group_col in group_cols:
                    if group_col in df.columns:
                        # Calculate group medians
                        group_medians = df.groupby(group_col)[col].transform('median')
                        # Fill missing values within groups
                        df[col] = df[col].fillna(group_medians)
            
            # Global fallback for any remaining missing values
            global_median = df[col].median()
            df[col] = df[col].fillna(global_median)
        else:
            # Use mode for categorical data
            if group_cols and len(group_cols) > 0:
                # Group-based imputation
                for group_col in group_cols:
                    if group_col in df.columns:
                        # Calculate group modes
                        group_modes = df.groupby(group_col)[col].transform(
                            lambda x: x.mode()[0] if not x.mode().empty else np.nan
                        )
                        # Fill missing values within groups
                        df[col] = df[col].fillna(group_modes)
            
            # Global fallback for any remaining missing values
            if df[col].mode().size > 0:
                global_mode = df[col].mode()[0]
                df[col] = df[col].fillna(global_mode)
    
    return df

# Apply the function
df = impute_missing(df)

# Check question: Mean of cargo_temp_f for hub_clean == "denver-east"
denver_east_temp = df[df['hub_clean'] == 'denver-east']['cargo_temp_f'].mean()
print(f"Question 14 Check: Mean cargo_temp_f for denver-east hub: {denver_east_temp:.2f}")
print(f"Missing values remaining in dataset: {df.isnull().sum().sum()}")


### Question 15

**Goal**: Create a function named cap_outliers_iqr(df, cols=None) that handles extreme values using a statistical outlier detection method.

Your function requirements:

* Accept flexible column specification:
  * If `cols=None` (default), process all numeric non-boolean columns
  * If `cols` is a string, process that single column
  * If `cols` is a list, process those specific columns

* Implement outlier detection and capping:
  * Use a robust statistical method based on the interquartile range (IQR)
  * Identify values that fall outside a reasonable range
  * Instead of removing outliers, adjust them to boundary values (winsorization)
  * This preserves all data rows while limiting extreme influence

* Apply Tukey's fence method:
  * Calculate quartiles and the interquartile range
  * Define bounds using a standard multiplier of the IQR
  * Cap values that exceed these bounds

* Preserve data integrity:
  * Never drop rows or columns
  * Ensure deterministic results

**Hints**:
- IQR (Interquartile Range) = Q3 - Q1
- Tukey's fences use the multiplier 1.5 for outlier detection
- Quartiles can be calculated with `.quantile()`
- "Capping" or "winsorization" means setting values to the boundary rather than removing them
- The `.clip()` method can limit values to a range

**Check question**: After running your function on the output of Question 14 with default parameters, what is the maximum value of service_time_min rounded to 4 decimal places?

In [ ]:
# Question 15

def cap_outliers_iqr(df, cols=None):
    """
    Handle extreme values using Tukey's fence method (IQR-based outlier detection).
    Caps outliers to boundary values (winsorization) instead of removing them.
    """
    df = df.copy()
    
    # If cols is None, process all numeric non-boolean columns
    if cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        # Exclude boolean-like columns
        cols = [col for col in numeric_cols 
               if df[col].dropna().nunique() > 2]
    elif isinstance(cols, str):
        cols = [cols]
    
    for col in cols:
        if col not in df.columns:
            continue
        
        # Calculate quartiles and IQR
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        # Define Tukey's fences with multiplier 1.5
        lower_fence = Q1 - 1.5 * IQR
        upper_fence = Q3 + 1.5 * IQR
        
        # Cap values to the fences (winsorization)
        df[col] = df[col].clip(lower=lower_fence, upper=upper_fence)
    
    return df

# Apply the function with default parameters
df = cap_outliers_iqr(df)

# Check question: Maximum value of service_time_min
max_service_time = df['service_time_min'].max()
print(f"Question 15 Check: Maximum service_time_min after capping: {max_service_time:.4f}")
print(f"Summary statistics for service_time_min:")
print(df['service_time_min'].describe())


### Question 16

**Goal**: Create a comprehensive business analysis report that demonstrates the value of data cleaning by revealing actionable insights about delivery performance.

**Task**: Using the fully cleaned dataset (from the prior question assuming it has undergone each of the prior steps), generate an analysis report with the following components:

1. **Hub Performance Analysis**
   * Calculate the delivery success rate (percentage of deliveries with status "delivered") for each hub
   * Sort hubs from highest to lowest success rate
   * Display results in a formatted table showing success_rate_pct for each hub_clean value

2. **Priority Level Risk Analysis**
   * Calculate the delivery failure rate (percentage of deliveries with status "failed") for each priority level
   * Sort priority levels from highest to lowest failure rate
   * Display results in a formatted table showing failure_rate_pct for each priority_level_clean value

3. **Visualization**
   * Create a 2-panel figure (1 row, 2 columns, size 14x5 inches) using matplotlib subplots
   * Left panel: Horizontal bar chart showing success rate by hub (steelblue color)
     - X-axis: "Success Rate (%)", Y-axis: "Hub"
     - Title: "Delivery Success Rate by Hub" (bold, size 13)
     - Add percentage labels on each bar
   * Right panel: Horizontal bar chart showing failure rate by priority (coral color)
     - X-axis: "Failure Rate (%)", Y-axis: "Priority Level"
     - Title: "Delivery Failure Rate by Priority" (bold, size 13)
     - Add percentage labels on each bar

4. **Business Insights Summary**
   * Print a formatted section titled "ACTIONABLE BUSINESS INSIGHTS" with:
   * **Hub Performance Analysis**: Identify best and worst performing hubs, calculate the performance gap, and recommend investigating underperforming locations
   * **Priority Level Risk Analysis**: Identify highest and lowest risk priority levels, calculate the risk differential, and recommend resource allocation strategies
   * **Data Cleaning Impact**: Explain how cleaning enabled this analysis (e.g., "Before cleaning: 14+ inconsistent hub names prevented reliable analysis. After cleaning: 3 standardized hubs enable actionable insights")

**Output Format**: Your code should print clearly formatted tables and insights using equal sign separators (60 characters) and emoji icons (📊, 📦, ✅) to organize the report sections.

**Hint**: You should be able to plug these requirements directly into an AI agent and it will write the code for you.

**Check question**: What is the delivery success rate (percentage of "delivered" status) for the hub with the highest success rate, rounded to 1 decimal place?

In [ ]:
# Question 16

# Business Analysis Report

print("=" * 60)
print("📊 DELIVERY PERFORMANCE ANALYSIS REPORT")
print("=" * 60)
print()

# 1. Hub Performance Analysis
print("=" * 60)
print("📦 HUB PERFORMANCE ANALYSIS")
print("=" * 60)

hub_stats = df.groupby('hub_clean')['delivery_status_clean'].apply(
    lambda x: (x == 'delivered').sum() / len(x) * 100
).sort_values(ascending=False)

hub_df = pd.DataFrame({
    'hub_clean': hub_stats.index,
    'success_rate_pct': hub_stats.values
})

print(hub_df.to_string(index=False))
print()

# 2. Priority Level Risk Analysis
print("=" * 60)
print("✅ PRIORITY LEVEL RISK ANALYSIS")
print("=" * 60)

priority_stats = df.groupby('priority_level_clean')['delivery_status_clean'].apply(
    lambda x: (x == 'failed').sum() / len(x) * 100
).sort_values(ascending=False)

priority_df = pd.DataFrame({
    'priority_level_clean': priority_stats.index,
    'failure_rate_pct': priority_stats.values
})

print(priority_df.to_string(index=False))
print()

# 3. Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Hub success rates
ax1 = axes[0]
hub_df_sorted = hub_df.sort_values('success_rate_pct')
bars1 = ax1.barh(hub_df_sorted['hub_clean'], hub_df_sorted['success_rate_pct'], 
                 color='steelblue')
ax1.set_xlabel('Success Rate (%)')
ax1.set_ylabel('Hub')
ax1.set_title('Delivery Success Rate by Hub', fontweight='bold', fontsize=13)

# Add percentage labels on bars
for i, bar in enumerate(bars1):
    width = bar.get_width()
    ax1.text(width, bar.get_y() + bar.get_height()/2, 
            f'{width:.1f}%', ha='left', va='center', fontsize=10)

# Right panel: Priority failure rates
ax2 = axes[1]
priority_df_sorted = priority_df.sort_values('failure_rate_pct')
bars2 = ax2.barh(priority_df_sorted['priority_level_clean'], 
                 priority_df_sorted['failure_rate_pct'], color='coral')
ax2.set_xlabel('Failure Rate (%)')
ax2.set_ylabel('Priority Level')
ax2.set_title('Delivery Failure Rate by Priority', fontweight='bold', fontsize=13)

# Add percentage labels on bars
for i, bar in enumerate(bars2):
    width = bar.get_width()
    ax2.text(width, bar.get_y() + bar.get_height()/2, 
            f'{width:.1f}%', ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('/Users/waylansmac/Desktop/455/delivery_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# 4. Business Insights Summary
print("=" * 60)
print("📊 ACTIONABLE BUSINESS INSIGHTS")
print("=" * 60)
print()

# Hub Performance Insights
best_hub = hub_df.iloc[0]
worst_hub = hub_df.iloc[-1]
performance_gap = best_hub['success_rate_pct'] - worst_hub['success_rate_pct']

print(f"**Hub Performance Analysis:**")
print(f"  - Best performing hub: {best_hub['hub_clean']} ({best_hub['success_rate_pct']:.1f}% success rate)")
print(f"  - Worst performing hub: {worst_hub['hub_clean']} ({worst_hub['success_rate_pct']:.1f}% success rate)")
print(f"  - Performance gap: {performance_gap:.1f} percentage points")
print(f"  - RECOMMENDATION: Investigate operational differences at {worst_hub['hub_clean']}.")
print(f"    Consider implementing best practices from {best_hub['hub_clean']}.")
print()

# Priority Level Risk Insights
highest_risk = priority_df.iloc[0]
lowest_risk = priority_df.iloc[-1]
risk_differential = highest_risk['failure_rate_pct'] - lowest_risk['failure_rate_pct']

print(f"**Priority Level Risk Analysis:**")
print(f"  - Highest risk priority: {highest_risk['priority_level_clean']} ({highest_risk['failure_rate_pct']:.1f}% failure rate)")
print(f"  - Lowest risk priority: {lowest_risk['priority_level_clean']} ({lowest_risk['failure_rate_pct']:.1f}% failure rate)")
print(f"  - Risk differential: {risk_differential:.1f} percentage points")
print(f"  - RECOMMENDATION: Allocate additional resources to {highest_risk['priority_level_clean']} deliveries")
print(f"    to reduce failure rates and improve customer satisfaction.")
print()

# Data Cleaning Impact
original_hub_count = df['hub'].nunique()
cleaned_hub_count = df['hub_clean'].nunique()

print(f"**Data Cleaning Impact:**")
print(f"  Before cleaning: {original_hub_count} inconsistent hub names prevented reliable analysis.")
print(f"  After cleaning: {cleaned_hub_count} standardized hubs enable actionable insights.")
print(f"  - Standardized {len(df)} delivery records across all cleaning steps")
print(f"  - Resolved missing values, outliers, and inconsistent formatting")
print(f"  - Created time-based features for temporal analysis")
print(f"  - Pipeline is reproducible and ready for production use")
print()

print("=" * 60)

# Check question for Question 16
best_success_rate = hub_df['success_rate_pct'].max()
print(f"\nQuestion 16 Check: Highest hub success rate: {best_success_rate:.1f}%")


In [ ]:
# Export Cleaned Dataset

# Save the fully cleaned dataset
output_path = '/Users/waylansmac/Desktop/455/cleaned_delivery_data.csv'
df.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset exported to: {output_path}")
print(f"   - Total rows: {len(df)}")
print(f"   - Total columns: {len(df.columns)}")
print(f"   - Missing values: {df.isnull().sum().sum()}")
print()
print("Dataset shape:", df.shape)
print("\nFirst few rows of cleaned data:")
print(df.head())
